# 🚀 Bane Agent: Neural DPO Benchmark (Google Colab / Kaggle T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pariveshkoshta-spec/Bane_Agent/blob/main/notebooks/Bane_Kaggle_GPU_Benchmark.ipynb)

This notebook runs our fine-tuned DPO LLaMA-3 model (`bane_dpo_lora_adapters`) with **full hardware acceleration** on an NVIDIA T4 GPU.
It benchmarks all **30 Enterprise Queries** (20 Technical + 10 Conversational Slang) against `enterprise_nexus.sqlite`.

### Step 1: Install High-Performance GPU Dependencies (Fast, Zero-RAM Spill)

In [ ]:
%%capture
# Fast pre-built wheel installation (prevents CPU/RAM spikes)
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets
!pip install faiss-cpu sentence-transformers rich fastapi uvicorn requests

### Step 2: Clone the Bane Agent Repository & Pull Code

In [ ]:
import os
import sys

# Clone repo if not present
if not os.path.exists("/kaggle/working/Bane_Agent") and not os.path.exists("Bane_Agent") and not os.path.exists("/content/Bane_Agent"):
    !git clone https://github.com/pariveshkoshta-spec/Bane_Agent.git

if os.path.exists("/kaggle/working/Bane_Agent"):
    %cd /kaggle/working/Bane_Agent
elif os.path.exists("/content/Bane_Agent"):
    %cd /content/Bane_Agent
elif os.path.exists("Bane_Agent"):
    %cd Bane_Agent

!git pull

### Step 3: Verify GPU Acceleration & LoRA Adapter Presence

In [ ]:
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU is OFF. Please switch runtime: Runtime -> Change runtime type -> T4 GPU")

# Check adapter presence in Colab or Kaggle
candidates = [
    "/content/bane_dpo_lora_adapters",
    "/kaggle/working/bane_dpo_lora_adapters",
    "/kaggle/working/outputs/checkpoint-375",
    "results/bane_dpo_lora_adapters",
    "bane_dpo_lora_adapters"
]
found_adapter = None
for c in candidates:
    if os.path.exists(c) and os.path.exists(os.path.join(c, "adapter_config.json")):
        found_adapter = c
        print(f"✅ Found fine-tuned adapters at: {c}")
        break

if not found_adapter:
    print("ℹ️ Note: If you haven't uploaded 'bane_dpo_lora_adapters' yet, simply drag-and-drop the folder into the file explorer on the left!")

### Step 4: Run the Full 30-Question GPU Benchmark!
Runs inference on the NVIDIA T4 GPU and executes every query against `enterprise_nexus.sqlite`.

In [ ]:
!python scripts/run_gpu_benchmark.py

### Step 5: View the Generated Benchmark Report & Gap Analysis

In [ ]:
from IPython.display import display, Markdown
if os.path.exists("KAGGLE_GPU_BENCHMARK_REPORT.md"):
    with open("KAGGLE_GPU_BENCHMARK_REPORT.md", "r") as f:
        report_content = f.read()
    display(Markdown(report_content[:4000] + "\n\n*(truncated for preview)*"))
else:
    print("Report not generated yet.")